# Lab 4 — Cluster Metrics

**Day 05 · Unsupervised Learning · Cisco AI/ML Training**

---

## Learning objectives

1. Evaluate K-Means clusters with **silhouette**, **Davies-Bouldin**, and **Calinski-Harabasz** scores.
2. Know which metrics are higher-is-better vs lower-is-better.
3. Interpret modest silhouette on real financial features (~0.24).
4. Compare metrics at k=**4** (Lab 1) vs k=**3** (Lab 2 elbow).

> **Checkpoints:** k=**4** · silhouette ≈ **0.24** · Davies-Bouldin ≈ **1.07** · Calinski-Harabasz ≈ **8.26**



## Clustering metrics (no ground truth)

Unlike Day 3 accuracy, we have **no labels** — metrics measure internal cluster quality.

| Metric | Direction | Intuition |
|--------|-----------|----------|
| **Silhouette** | Higher better (max 1) | How similar a point is to its cluster vs neighbors |
| **Davies-Bouldin** | Lower better | Average similarity between each cluster and its closest match |
| **Calinski-Harabasz** | Higher better | Ratio of between-cluster to within-cluster dispersion |

Silhouette ≈ **0.24** on 25 NYSE symbols is expected — financial features do not form perfectly separated spheres.

---

## 1. Load features and fit K-Means (k = 4)

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,
)
from sklearn.preprocessing import StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-05":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "nyse" / "nyse_stocks.csv").is_file():
            GH_ROOT = parent
            break

FEATURE_COLUMNS = ["avg_close", "volatility", "avg_volume", "avg_range"]

nyse = pd.read_csv(GH_ROOT / "data" / "nyse" / "nyse_stocks.csv", parse_dates=["date"])
nyse["range"] = nyse["high"] - nyse["low"]
features = (
    nyse.groupby("symbol")
    .agg(
        avg_close=("close", "mean"),
        volatility=("close", "std"),
        avg_volume=("volume", "mean"),
        avg_range=("range", "mean"),
    )
    .reset_index()
)
features["volatility"] = features["volatility"].fillna(0.0)

X_scaled = StandardScaler().fit_transform(features[FEATURE_COLUMNS])

k = 4
labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
print(f"symbols: {len(features)}, k: {k}")

---

## 2. Compute all three metrics

In [ ]:
sil = silhouette_score(X_scaled, labels)
db = davies_bouldin_score(X_scaled, labels)
ch = calinski_harabasz_score(X_scaled, labels)

print("Lab 4 — Cluster metrics")
print(f"k: {k}")
print(f"silhouette score: {sil:.4f}  (higher is better, max 1)")
print(f"Davies-Bouldin index: {db:.4f}  (lower is better)")
print(f"Calinski-Harabasz score: {ch:.4f}  (higher is better)")

metrics_df = pd.DataFrame({
    "metric": ["silhouette", "davies_bouldin", "calinski_harabasz"],
    "value": [sil, db, ch],
    "direction": ["higher better", "lower better", "higher better"],
})
display(metrics_df.round(4))

### Silhouette in plain language

For each symbol, silhouette compares **cohesion** (distance to own cluster) vs **separation** (distance to nearest other cluster). Values near **0** mean overlapping clusters; near **1** mean well-separated.

---

## 3. Per-point silhouette (optional deep dive)

In [ ]:
from sklearn.metrics import silhouette_samples

sample_scores = silhouette_samples(X_scaled, labels)
features = features.copy()
features["cluster"] = labels
features["silhouette"] = sample_scores.round(3)

display(
    features[["symbol", "cluster", "silhouette"]]
    .sort_values("silhouette")
    .head(5)
)

Lowest silhouette symbols sit near cluster boundaries — candidates for DBSCAN noise (Lab 3).

---

## 4. Compare k = 3 vs k = 4 (Lab 2 elbow)

In [ ]:
rows = []
for k_try in [3, 4]:
    lbl = KMeans(n_clusters=k_try, random_state=42, n_init=10).fit_predict(X_scaled)
    rows.append({
        "k": k_try,
        "silhouette": silhouette_score(X_scaled, lbl),
        "davies_bouldin": davies_bouldin_score(X_scaled, lbl),
        "calinski_harabasz": calinski_harabasz_score(X_scaled, lbl),
    })

compare_k = pd.DataFrame(rows)
display(compare_k.round(4))

Elbow (Lab 2) and silhouette may disagree — use multiple criteria plus business judgment.

---

## 5. Visualize metrics at k = 4

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_df = metrics_df.copy()
plot_df["display_value"] = plot_df["value"] / plot_df["value"].max()
sns.barplot(data=plot_df, x="metric", y="display_value", ax=ax, palette="Set2")
ax.set_ylabel("normalized value (for display)")
ax.set_title(f"Cluster metrics (k={k}) — check direction column for interpretation")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()

---

## 6. Checkpoint summary

In [ ]:
assert k == 4
assert abs(sil - 0.2414) < 0.05
assert abs(db - 1.0659) < 0.1
assert abs(ch - 8.2627) < 1.0
print("✓ All checkpoint assertions passed")

---

## Reflection questions

1. Why is silhouette lower here than typical textbook examples (~0.7+)?
2. Can you apply these metrics to DBSCAN when label **-1** (noise) exists?
3. Which metric would you report to a non-technical stakeholder and why?

**Previous:** [Lab 3 — DBSCAN clusters](lab03_dbscan_clusters.ipynb)  
**Next:** [Lab 5 — NYSE multi-cluster view](lab05_nyse_multi_cluster_view.ipynb)